# Model 2: Skin Diseases Dataset Training

This notebook trains a ResNet152V2 model on the Skin Diseases dataset (ismailpromus).
**Optimized for Google Colab T4 GPU**

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
MODEL_SAVE_DIR = '/content/drive/MyDrive/CMPE295 Project/ML/Ensemble/Models'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
!pip install kagglehub -q

import kagglehub
import numpy as np
import json
import shutil
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.applications import ResNet152V2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Download Skin Diseases dataset
print("Downloading Skin Diseases Dataset...")
SKIN_PATH = kagglehub.dataset_download("ismailpromus/skin-diseases-image-dataset")
print(f"Dataset Path: {SKIN_PATH}")

Using Colab cache for faster access to the 'skin-diseases-image-dataset' dataset.
Dataset Path: /kaggle/input/skin-diseases-image-dataset


In [ ]:
# Navigate to IMG_CLASSES
IMG_CLASSES_PATH = os.path.join(SKIN_PATH, 'IMG_CLASSES')

print("Classes found:")
for cls in os.listdir(IMG_CLASSES_PATH):
    cls_path = os.path.join(IMG_CLASSES_PATH, cls)
    if os.path.isdir(cls_path):
        num_imgs = len([f for f in os.listdir(cls_path) if f.endswith('.jpg')])
        print(f"  {cls}: {num_imgs} images")

Classes found:
  1. Eczema 1677: 1677 images
  10. Warts Molluscum and other Viral Infections - 2103: 2103 images
  4. Basal Cell Carcinoma (BCC) 3323: 3323 images
  7. Psoriasis pictures Lichen Planus and related diseases - 2k: 2055 images
  5. Melanocytic Nevi (NV) - 7970: 7970 images
  9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k: 1702 images
  3. Atopic Dermatitis - 1.25k: 1257 images
  6. Benign Keratosis-like Lesions (BKL) 2624: 2079 images
  8. Seborrheic Keratoses and other Benign Tumors - 1.8k: 1847 images
  2. Melanoma 15.75k: 3140 images


In [ ]:
# Create train/test split
WORK_DIR = '/content/skin_diseases_organized'
TRAIN_DIR = os.path.join(WORK_DIR, 'train')
TEST_DIR = os.path.join(WORK_DIR, 'test')

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

for cls_folder in tqdm(os.listdir(IMG_CLASSES_PATH), desc="Organizing images"):
    cls_path = os.path.join(IMG_CLASSES_PATH, cls_folder)

    if not os.path.isdir(cls_path):
        continue

    # Clean class name
    clean_name = cls_folder.split('. ', 1)[-1] if '. ' in cls_folder else cls_folder

    # Create class folders
    os.makedirs(os.path.join(TRAIN_DIR, clean_name), exist_ok=True)
    os.makedirs(os.path.join(TEST_DIR, clean_name), exist_ok=True)

    # Get all images
    images = [f for f in os.listdir(cls_path) if f.endswith('.jpg')]

    # Split
    train_imgs, test_imgs = train_test_split(images, test_size=0.2, random_state=42)

    # Copy train
    for img in train_imgs:
        src = os.path.join(cls_path, img)
        dst = os.path.join(TRAIN_DIR, clean_name, img)
        shutil.copy2(src, dst)

    # Copy test
    for img in test_imgs:
        src = os.path.join(cls_path, img)
        dst = os.path.join(TEST_DIR, clean_name, img)
        shutil.copy2(src, dst)

print("Dataset organized!")

Organizing images: 100%|██████████| 10/10 [05:43<00:00, 34.38s/it]

Dataset organized!


In [ ]:
# Setup generators
IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)
print(f"Classes: {class_names}")

Found 21719 images belonging to 10 classes.
Found 5434 images belonging to 10 classes.
Classes: ['Atopic Dermatitis - 1.25k', 'Basal Cell Carcinoma (BCC) 3323', 'Benign Keratosis-like Lesions (BKL) 2624', 'Eczema 1677', 'Melanocytic Nevi (NV) - 7970', 'Melanoma 15.75k', 'Psoriasis pictures Lichen Planus and related diseases - 2k', 'Seborrheic Keratoses and other Benign Tumors - 1.8k', 'Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k', 'Warts Molluscum and other Viral Infections - 2103']


In [ ]:
# Build model
base_model = ResNet152V2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

234545216/234545216 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step


In [ ]:
# Callbacks
callbacks = [
    ModelCheckpoint(
        filepath=os.path.join(MODEL_SAVE_DIR, 'skin_diseases_best.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3
    )
]

In [ ]:
# Train Phase 1
history1 = model.fit(
    train_generator,
    epochs=10,
    validation_data=test_generator,
    callbacks=callbacks
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
679/679 ━━━━━━━━━━━━━━━━━━━━ 0s 652ms/step - accuracy: 0.4334 - loss: 1.6824
Epoch 1: val_accuracy improved from -inf to 0.62017, saving model to /content/drive/MyDrive/CMPE295 Project/ML/Ensemble/Models/skin_diseases_best.keras
679/679 ━━━━━━━━━━━━━━━━━━━━ 546s 753ms/step - accuracy: 0.4335 - loss: 1.6820 - val_accuracy: 0.6202 - val_loss: 0.9954 - learning_rate: 0.0010
Epoch 2/10
679/679 ━━━━━━━━━━━━━━━━━━━━ 0s 625ms/step - accuracy: 0.5511 - loss: 1.1486
Epoch 2: val_accuracy improved from 0.62017 to 0.63287, saving model to /content/drive/MyDrive/CMPE295 Project/ML/Ensemble/Models/skin_diseases_best.keras
679/679 ━━━━━━━━━━━━━━━━━━━━ 476s 700ms/step - accuracy: 0.5511 - loss: 1.1486 - val_accuracy: 0.6329 - val_loss: 0.9807 - learning_rate: 0.0010
Epoch 3/10
679/679 ━━━━━━━━━━━━━━━━━━━━ 0s 617ms/step - accuracy: 0.5879 - loss: 1.0741
Epoch 3: val_accuracy improved from 0.63287 to 0.64649, saving model to /content/drive/MyDrive/CMPE295 Project/ML/Ensemble/Models/skin_dise

In [ ]:
# Train Phase 2
base_model.trainable = True
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_generator,
    epochs=10,
    validation_data=test_generator,
    callbacks=callbacks
)

Epoch 1/10
679/679 ━━━━━━━━━━━━━━━━━━━━ 0s 906ms/step - accuracy: 0.6034 - loss: 1.0429
Epoch 1: val_accuracy improved from 0.66949 to 0.68568, saving model to /content/drive/MyDrive/CMPE295 Project/ML/Ensemble/Models/skin_diseases_best.keras
679/679 ━━━━━━━━━━━━━━━━━━━━ 834s 1s/step - accuracy: 0.6035 - loss: 1.0428 - val_accuracy: 0.6857 - val_loss: 0.8496 - learning_rate: 1.0000e-04
Epoch 2/10
679/679 ━━━━━━━━━━━━━━━━━━━━ 0s 875ms/step - accuracy: 0.6786 - loss: 0.8553
Epoch 2: val_accuracy did not improve from 0.68568
679/679 ━━━━━━━━━━━━━━━━━━━━ 643s 947ms/step - accuracy: 0.6786 - loss: 0.8553 - val_accuracy: 0.6649 - val_loss: 0.8979 - learning_rate: 1.0000e-04
Epoch 3/10
679/679 ━━━━━━━━━━━━━━━━━━━━ 0s 850ms/step - accuracy: 0.7089 - loss: 0.7879
Epoch 3: val_accuracy did not improve from 0.68568
679/679 ━━━━━━━━━━━━━━━━━━━━ 627s 923ms/step - accuracy: 0.7089 - loss: 0.7879 - val_accuracy: 0.6308 - val_loss: 1.0254 - learning_rate: 1.0000e-04
Epoch 4/10
679/679 ━━━━━━━━━━━━━━━━

In [ ]:
# Evaluate and save
test_loss, test_accuracy = model.evaluate(test_generator)
print(f"\nTest Accuracy: {test_accuracy*100:.2f}%")

model.save(os.path.join(MODEL_SAVE_DIR, 'skin_diseases_final.keras'))

with open(os.path.join(MODEL_SAVE_DIR, 'skin_diseases_classes.json'), 'w') as f:
    json.dump(class_names, f)

model_info = {
    'dataset': 'Skin Diseases (ismailpromus)',
    'num_classes': num_classes,
    'classes': class_names,
    'test_accuracy': float(test_accuracy),
    'image_type': 'clinical',
    'input_size': IMG_SIZE
}

with open(os.path.join(MODEL_SAVE_DIR, 'skin_diseases_info.json'), 'w') as f:
    json.dump(model_info, f, indent=2)

print("\n✅ Model saved to Google Drive!")

NameError: name 'model' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import tensorflow as tf
from tensorflow import keras
from google.colab import files

model.save('skin_diseases.h5')
files.download('skin_diseases.h5')

NameError: name 'model' is not defined